# 05 — Bias & Fairness Audit

**Hospital Readmission Risk Predictor**

Healthcare AI models can inadvertently perpetuate or amplify existing disparities. This notebook
audits model performance across race, gender, and age subgroups to identify any inequitable
patterns before clinical deployment.

A model that performs well on average but poorly for specific demographic groups is not
acceptable in a healthcare context.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_and_prepare
from src.leakage_detection import remove_leakage_rows
from src.feature_engineering import engineer_features
from src.preprocessing import split_X_y
from src.fairness import (
    run_fairness_audit, detect_significant_gaps,
    generate_fairness_report, plot_fairness_comparison
)
from src.utils import set_plot_style, load_pipeline, set_seed
from src.config import RANDOM_SEED, TEST_SIZE

from sklearn.model_selection import train_test_split

set_plot_style()
set_seed()
print('Setup complete ✓')

In [ ]:
# Load data and model
df = load_and_prepare()
df = remove_leakage_rows(df)
df = engineer_features(df)

X, y = split_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)

pipeline = load_pipeline()
y_prob = pipeline.predict_proba(X_test)[:, 1]
threshold = 0.3
y_pred = (y_prob >= threshold).astype(int)

print(f'Test set: {len(X_test):,} patients')
print(f'Predictions — flagged: {y_pred.sum():,} ({y_pred.mean()*100:.1f}%)')

## 1. Subgroup Performance Analysis

We compute precision, recall, F1, false positive rate (FPR), and false negative rate (FNR)
for every subgroup defined by race, gender, and age.

In [ ]:
audit_df = run_fairness_audit(X_test, y_test.values, y_pred)
display(audit_df)

## 2. Gap Detection

We automatically flag any subgroup pair where the recall or precision gap exceeds 10%.
A gap of this magnitude could mean that the model is systematically under-serving certain
patient populations.

In [ ]:
gaps_df = detect_significant_gaps(audit_df)

if len(gaps_df) > 0:
    print('⚠ Significant fairness gaps detected:')
    display(gaps_df)
else:
    print('✓ No gaps exceeding 10% detected across subgroups.')

## 3. Visual Comparison

In [ ]:
for attr in ['race', 'gender', 'age']:
    if attr in audit_df['attribute'].values:
        plot_fairness_comparison(audit_df, attribute=attr)

## 4. Fairness Discussion

### What We Observe

Readmission prediction in healthcare is inherently challenging from a fairness perspective
because the data itself reflects systemic inequities:

- **Access disparities** — Patients with better insurance and closer proximity to care
  facilities have different utilisation patterns, which the model may interpret as lower risk.

- **Documentation bias** — Clinical documentation quality varies across hospitals and
  populations, affecting the signal available in diagnosis codes and procedures.

- **Sample size effects** — Minority groups have fewer samples, leading to wider confidence
  intervals around their performance metrics.

### What We Recommend

1. **Do not deploy without monitoring** — Set up automated alerts for subgroup metric drift.
2. **Consider per-group thresholds** — Different operating points for different populations
   may improve equity, though this raises its own ethical questions.
3. **Engage stakeholders** — Clinical ethicists, patient advocates, and community
   representatives should review these findings before deployment.
4. **Augment with qualitative data** — Social determinants of health (housing stability,
   caregiver availability) may help close performance gaps.
5. **Iterative improvement** — Fairness is not a one-time checkbox but an ongoing commitment.

In [ ]:
# Generate the full fairness report
report_path = generate_fairness_report(audit_df, gaps_df)
print(f'\nFairness report saved: {report_path}')

## Summary

This audit provides transparency about how the model performs across demographic groups.
Whether or not significant gaps were detected, the audit itself demonstrates a commitment
to responsible AI in healthcare — a commitment that must extend beyond model development
into deployment, monitoring, and continuous improvement.